# Notebook 03 — Preprocessing
## Tiền xử lý dữ liệu, Pipeline, SMOTE

**Mục tiêu:**
1. Xây dựng sklearn Pipeline (StandardScaler)
2. Train-test split stratified (80/20)
3. Áp dụng SMOTE trên train set để xử lý class imbalance
4. Lưu processed data và pipeline


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.preprocess import run_preprocessing, build_pipeline, split_and_resample
from src.data.load_data import load_raw_data, encode_target, get_X_y, FEATURE_COLS

plt.rcParams['figure.dpi'] = 120
DATA_PATH = '../data/raw/dataset.csv'
print('Import thành công ✅')

## 1. Load và encode data

In [ ]:
df = load_raw_data(DATA_PATH)
df_enc = encode_target(df)
X, y = get_X_y(df_enc)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Phân bố nhãn: {np.bincount(y)}')

## 2. Pipeline — StandardScaler

In [ ]:
pipeline = build_pipeline()
print('Pipeline:')
print(pipeline)
print()
print('Giải thích: Dataset UCI đã mã hóa số sẵn, chỉ cần StandardScaler.')
print('Không cần One-Hot Encoding vì tất cả biến phân loại đã là số.')

## 3. Train-test split + SMOTE

In [ ]:
X_train, X_test, y_train, y_test, pipeline_fit = split_and_resample(
    X, y, test_size=0.2, use_smote=True
)

print(f'\nSau split và SMOTE:')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape},  y_test : {y_test.shape}')

In [ ]:
# So sánh phân bố trước và sau SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trước SMOTE (y gốc)
from sklearn.model_selection import train_test_split
_, __, y_train_orig, ___ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
counts_orig = np.bincount(y_train_orig)
counts_smote = np.bincount(y_train)
labels = ['Dropout', 'Enrolled', 'Graduate']
colors = ['#e74c3c', '#f39c12', '#2ecc71']

axes[0].bar(labels, counts_orig, color=colors, edgecolor='white')
for i, v in enumerate(counts_orig):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=10)
axes[0].set_title('Trước SMOTE', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Số mẫu')

axes[1].bar(labels, counts_smote, color=colors, edgecolor='white')
for i, v in enumerate(counts_smote):
    axes[1].text(i, v + 10, str(v), ha='center', fontsize=10)
axes[1].set_title('Sau SMOTE', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Số mẫu')

plt.suptitle('Phân bố nhãn trước/sau SMOTE (Train set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Kiểm tra scale

In [ ]:
print('Kiểm tra giá trị sau StandardScaler (trên X_train):')
df_scaled = pd.DataFrame(X_train, columns=FEATURE_COLS)
stats = df_scaled.describe().loc[['mean', 'std']].T
print(f'Mean ~ 0: {stats["mean"].abs().max():.4f} (nên ≈ 0)')
print(f'Std  ~ 1: {stats["std"].mean():.4f} (nên ≈ 1)')
print()
print('✅ Scale thành công' if stats['mean'].abs().max() < 0.01 else '⚠️ Kiểm tra lại scale')

## 5. Lưu processed data và pipeline

In [ ]:
from src.data.preprocess import save_processed_data, save_pipeline

save_processed_data(X_train, X_test, y_train, y_test, FEATURE_COLS, output_dir='../data/processed')
save_pipeline(pipeline_fit, path='../models/preprocessing_pipeline.pkl')

print('\nChecklist:')
import os
files_to_check = [
    '../data/processed/X_train.csv',
    '../data/processed/X_test.csv',
    '../data/processed/y_train.csv',
    '../data/processed/y_test.csv',
    '../models/preprocessing_pipeline.pkl',
]
for f in files_to_check:
    status = '✅' if os.path.exists(f) else '❌'
    print(f'  {status} {f}')